# Práctica 1: Fundamentos de Probabilidad e Interactividad Bayesiana

¡Bienvenido a la primera práctica del curso! En esta libreta exploraremos de forma visual e interactiva los conceptos clave de probabilidad y estadística que sustentan la estimación probabilística de estados.

### Objetivos:
1. Comprender el impacto de la media y la varianza en la distribución Gaussiana (1D y 2D).
2. Implementar paso a paso el **Teorema de Bayes** para un caso discreto (el problema de localización en el pasillo).
3. Programar la **multiplicación de dos distribuciones Gaussianas** (fusión de información), sentando las bases analíticas del Filtro de Kalman.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, multivariate_normal
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

# Configuración de estilo premium para las gráficas
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['figure.titlesize'] = 16

---
## 1. La Campana de Gauss en 1D

La distribución Gaussiana univariada está completamente descrita por su media ($\mu$) y su desviación estándar ($\sigma$). Observemos interactuando cómo el cambio en estos parámetros deforma la PDF.

In [2]:
def plot_gaussiana_1d(mu, sigma):
    x = np.linspace(-10, 10, 1000)
    y = norm.pdf(x, loc=mu, scale=sigma)
    
    plt.figure(figsize=(10, 5))
    plt.plot(x, y, label=f'$\mathcal{{N}}(\mu={mu}, \sigma^2={sigma**2:.2f})$', color='#1f77b4', linewidth=2.5)
    plt.fill_between(x, y, alpha=0.15, color='#1f77b4')
    
    # Línea de la media
    plt.axvline(mu, color='red', linestyle='--', alpha=0.7, label=f'Media ($\mu$) = {mu}')
    
    # Líneas de desviación estándar (1-sigma)
    plt.axvline(mu - sigma, color='orange', linestyle=':', alpha=0.7, label=f'$\mu \pm \sigma$')
    plt.axvline(mu + sigma, color='orange', linestyle=':', alpha=0.7)
    
    plt.title("Distribución Gaussiana Univariada", pad=15)
    plt.xlabel("Valor de la Variable Aleatoria ($x$)")
    plt.ylabel("Densidad de Probabilidad ($p(x)$)")
    plt.xlim(-10, 10)
    plt.ylim(0, 1.0)
    plt.legend(frameon=True, facecolor='white', framealpha=0.9)
    plt.show()

# Crear widget interactivo
interact(plot_gaussiana_1d, 
         mu=FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.5, description='Media ($\mu$):'),
         sigma=FloatSlider(value=1.0, min=0.4, max=3.0, step=0.1, description='Desv. Est. ($\sigma$):'));

<>:6: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<>:6: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<>:6: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:10: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<>:13: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<>:26: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<>:27: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will n

interactive(children=(FloatSlider(value=0.0, description='Media ($\\mu$):', max=5.0, min=-5.0, step=0.5), Floa…

---
## 2. La Gaussiana Multivariada (2D)

En 2D, la campana de Gauss es tridimensional. Al proyectar sus contornos en el plano $x-y$, obtenemos **elipses de incertidumbre**. El comportamiento y orientación de la elipse están determinados por la **Matriz de Covarianza**:
$$\mathbf{\Sigma} = \begin{bmatrix} \sigma_x^2 & \sigma_{xy} \\ \sigma_{xy} & \sigma_y^2 \end{bmatrix}$$

Donde:
- $\sigma_x^2$ y $\sigma_y^2$ son las varianzas en los ejes X e Y.
- $\sigma_{xy}$ es la covarianza (relación de dependencia).

In [ ]:
def plot_gaussiana_2d(var_x, var_y, cov_xy):
    # Crear matriz de covarianza y comprobar que sea válida (semidefinida positiva)
    if cov_xy**2 >= var_x * var_y:
        # Ajustar para mantener la consistencia matemática
        cov_xy = np.sign(cov_xy) * np.sqrt(var_x * var_y) * 0.99
        print(f"⚠️ Ajustando covarianza a {cov_xy:.2f} para mantener consistencia física.")
        
    mu = np.array([0.0, 0.0])
    Sigma = np.array([
        [var_x, cov_xy],
        [cov_xy, var_y]
    ])
    
    # Malla para evaluar la distribución
    x, y = np.mgrid[-4:4:.01, -4:4:.01]
    pos = np.dstack((x, y))
    
    rv = multivariate_normal(mu, Sigma)
    z = rv.pdf(pos)
    
    fig, ax = plt.subplots(figsize=(8, 7))
    # Contornos llenos
    contour = ax.contourf(x, y, z, levels=15, cmap='viridis', alpha=0.85)
    fig.colorbar(contour, ax=ax, label='Densidad de Probabilidad $p(x, y)$')
    
    # Elipses de contorno discretos (1-sigma, 2-sigma, 3-sigma aproximados)
    ax.contour(x, y, z, levels=3, colors='white', linewidths=1.2, linestyles='--')
    
    ax.axhline(0, color='white', linestyle='-', alpha=0.3)
    ax.axvline(0, color='white', linestyle='-', alpha=0.3)
    
    ax.set_title("Proyección de Contorno de Gaussiana Bivariada (2D)", pad=15)
    ax.set_xlabel("Variable $X$")
    ax.set_ylabel("Variable $Y$")
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_aspect('equal')
    plt.show()

interact(plot_gaussiana_2d,
         var_x=FloatSlider(value=1.0, min=0.2, max=2.0, step=0.1, description='Varianza X ($\sigma_x^2$):'),
         var_y=FloatSlider(value=1.0, min=0.2, max=2.0, step=0.1, description='Varianza Y ($\sigma_y^2$):'),
         cov_xy=FloatSlider(value=0.0, min=-1.5, max=1.5, step=0.1, description='Covarianza XY ($\sigma_{xy}$):'));

---
## 3. Teorema de Bayes Discreto: Localización en Pasillo

Implementemos la lógica del ejercicio del pasillo. Supongamos que el robot sabe que está en un espacio discreto con dos posibles estados: **Pared** o **Puerta**.

El robot posee un sensor que mide la presencia de puertas. Programemos la función de actualización bayesiana.

In [ ]:
def actualizar_bayes(prior_puerta, likelihood_puerta_dado_puerta, likelihood_puerta_dado_pared):
    # 1. Definir prior complementario
    prior_pared = 1.0 - prior_puerta
    
    # Likelihood complementarios (el sensor detecta "Puerta")
    # P(z="Puerta"|x=Puerta)
    p_z_puerta = likelihood_puerta_dado_puerta
    # P(z="Puerta"|x=Pared)
    p_z_pared = likelihood_puerta_dado_pared
    
    # 2. Paso de actualización (Regla del Producto: Prior * Likelihood)
    unnormalized_puerta = prior_puerta * p_z_puerta
    unnormalized_pared = prior_pared * p_z_pared
    
    # 3. Normalizador (Evidencia total del sensor P(z="Puerta"))
    evidencia = unnormalized_puerta + unnormalized_pared
    
    # 4. Posterior
    posterior_puerta = unnormalized_puerta / evidencia
    posterior_pared = unnormalized_pared / evidencia
    
    # Gráfico de barras comparativo
    categorias = ['Puerta', 'Pared']
    priors = [prior_puerta, prior_pared]
    posteriors = [posterior_puerta, posterior_pared]
    
    x_indices = np.arange(len(categorias))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(x_indices - width/2, priors, width, label='Prior $P(X)$', color='#aec7e8')
    ax.bar(x_indices + width/2, posteriors, width, label='Posterior $P(X \mid z=\text{"Puerta"})$', color='#1f77b4')
    
    ax.set_ylabel('Probabilidad')
    ax.set_title('Actualización Bayesiana al detectar "Puerta"', pad=15)
    ax.set_xticks(x_indices)
    ax.set_xticklabels(categorias)
    ax.set_ylim(0, 1.05)
    ax.legend(frameon=True, facecolor='white')
    
    print(f"--- RESULTADOS DE LA ACTUALIZACIÓN ---")
    print(f"Prior de Puerta: {prior_puerta*100:.1f}%")
    print(f"Likelihood Verdadero Positivo: {p_z_puerta*100:.1f}%")
    print(f"Likelihood Falso Positivo (falsa detección en pared): {p_z_pared*100:.1f}%")
    print(f"-------------------------------------")
    print(f"Probabilidad Posterior de estar en PUERTA: {posterior_puerta*100:.2f}%")
    print(f"Probabilidad Posterior de estar en PARED:  {posterior_pared*100:.2f}%")
    
    plt.show()

interact(actualizar_bayes,
         prior_puerta=FloatSlider(value=0.20, min=0.01, max=0.99, step=0.05, description='Prior Puerta:'),
         likelihood_puerta_dado_puerta=FloatSlider(value=0.90, min=0.50, max=0.99, step=0.05, description='VP Sensor (P|Puerta):'),
         likelihood_puerta_dado_pared=FloatSlider(value=0.10, min=0.01, max=0.50, step=0.05, description='FP Sensor (P|Pared):'));

---
## 4. Multiplicación de Gaussianas (Fusión de Sensores Continua)

¿Qué pasa cuando la creencia es continua y Gaussiana? 
Supongamos que nuestro robot tiene una estimación de su posición (Prior) dada por:
$$\text{Prior} \sim \mathcal{N}(\mu_1, \sigma_1^2)$$

Y recibe una medición del sensor con un modelo de observación ruidoso (Likelihood) dada por:
$$\text{Medición} \sim \mathcal{N}(\mu_2, \sigma_2^2)$$

Si multiplicamos ambas gaussianas (fusión recursiva de Bayes), obtenemos otra distribución Gaussiana $\mathcal{N}(\mu_3, \sigma_3^2)$ donde:
$$\sigma_3^2 = \frac{1}{\frac{1}{\sigma_1^2} + \frac{1}{\sigma_2^2}} = \frac{\sigma_1^2 \sigma_2^2}{\sigma_1^2 + \sigma_2^2}$$

$$\mu_3 = \sigma_3^2 \left( \frac{\mu_1}{\sigma_1^2} + \frac{\mu_2}{\sigma_2^2} \right) = \mu_1 + \frac{\sigma_1^2}{\sigma_1^2 + \sigma_2^2}(\mu_2 - \mu_1)$$

¡Nota esta última ecuación! Es la estructura matemática exacta de la **Ecuación de Actualización del Filtro de Kalman**, donde el término $\frac{\sigma_1^2}{\sigma_1^2 + \sigma_2^2}$ se conoce como la **Ganancia de Kalman ($K$)**.

Programemos esta fusión y analicemos visualmente el comportamiento.

In [ ]:
def fusion_gaussianas(mu_prior, sigma_prior, mu_medida, sigma_medida):
    # Calcular varianzas
    var_prior = sigma_prior ** 2
    var_medida = sigma_medida ** 2
    
    # Calcular ganancia de Kalman (K)
    K = var_prior / (var_prior + var_medida)
    
    # Nuevos parámetros del posterior
    mu_posterior = mu_prior + K * (mu_medida - mu_prior)
    var_posterior = (1 - K) * var_prior
    sigma_posterior = np.sqrt(var_posterior)
    
    # Gráfico
    x = np.linspace(-15, 15, 1000)
    y_prior = norm.pdf(x, mu_prior, sigma_prior)
    y_medida = norm.pdf(x, mu_medida, sigma_medida)
    y_posterior = norm.pdf(x, mu_posterior, sigma_posterior)
    
    plt.figure(figsize=(12, 6))
    plt.plot(x, y_prior, label=f'Prior: $\mathcal{{N}}({mu_prior}, {var_prior:.2f})$', color='#d62728', linestyle='--', linewidth=2)
    plt.plot(x, y_medida, label=f'Medición: $\mathcal{{N}}({mu_medida}, {var_medida:.2f})$', color='#2ca02c', linestyle=':', linewidth=2)
    plt.plot(x, y_posterior, label=f'Posterior (Fusión): $\mathcal{{N}}({mu_posterior:.2f}, {var_posterior:.2f})$', color='#1f77b4', linewidth=3)
    
    plt.fill_between(x, y_posterior, alpha=0.15, color='#1f77b4')
    
    plt.title("Fusión Bayesiana de Dos Distribuciones Gaussianas", pad=15)
    plt.xlabel("Posición del Robot")
    plt.ylabel("Densidad de Probabilidad")
    plt.xlim(-12, 12)
    plt.ylim(0, 1.2)
    plt.legend(frameon=True, facecolor='white', framealpha=0.9)
    
    print(f"--- ANÁLISIS DE FUSIÓN ---")
    print(f"Ganancia de Kalman (K): {K:.4f}")
    print(f"(Si K es alto, confiamos más en la medición. Si es bajo, confiamos más en el prior)")
    print(f"-------------------------")
    print(f"Nueva Media Posterior: {mu_posterior:.4f}")
    print(f"Nueva Desviación Estándar: {sigma_posterior:.4f} (Varianza: {var_posterior:.4f})")
    print(f"¡Nota que la nueva varianza es MENOR que la del prior ({var_prior:.2f}) y de la medida ({var_medida:.2f})!")
    print(f"La certidumbre siempre aumenta al fusionar información.")
    
    plt.show()

interact(fusion_gaussianas,
         mu_prior=FloatSlider(value=-2.0, min=-8.0, max=8.0, step=0.5, description='Media Prior:'),
         sigma_prior=FloatSlider(value=1.5, min=0.5, max=4.0, step=0.1, description='σ Prior:'),
         mu_medida=FloatSlider(value=3.0, min=-8.0, max=8.0, step=0.5, description='Media Medida:'),
         sigma_medida=FloatSlider(value=1.0, min=0.5, max=4.0, step=0.1, description='σ Medida:'));